# Pre-JcvPCA Segment / Joint Review (Layer 2.5 control interface)

This notebook guides you from **participant/session selection** to a **canonically aligned, QC-aware, Layer 3-ready export**. The export path is **canonical only** — feature names are `parent_to_child_axis` (e.g. `Neck_to_Head_rx`), never legacy `J00x` names — and unsafe exports are **blocked**.

**Workflow**
1. **Input roots** — defaults are `Layer1_motive_qc/motive_qc/outputs` and `Layer2_Motive_Kinematics/outputs`. Click **Discover participants/sessions** (writes `session_index.csv`).
2. **Participant & session** — pick a participant. The notebook shows Layer 1 / Layer 2 **pairing status** and a **joint/link overlap & comparability table** (`joint_overlap_table.csv`), then pick a session.
3. **Window & feature scope** — set frame window + label. The default feature scope (core links, fingers/toes excluded) is controlled by `config/default_feature_scope.yaml`.
4. **Warnings** — click **Preview warnings** to see blocking/strong/info alerts *before* export.
5. **Export** — enabled only when blocking checks pass; writes the canonical `window_jvcpca_matrix.parquet`, `window_warnings.csv`, `window_export_manifest.json` (with `layer3_safe=true`), and a summary.

**Note:** Diagnostics (Run mapping / Run full review / Show review tables) write to
`outputs/pre_jvcpca_review/<participant>/<session>/<window_label>/` — the same window folder used for Layer 3 export. Run **Discover** and select a matched session before using them.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "pre_jvcpca_review").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import ipywidgets as widgets
import pandas as pd
from IPython.display import HTML, display

from pre_jvcpca_review.discovery import resolve_layer2
from pre_jvcpca_review.export_window import (
    MATRIX_IDENTITY_COLUMNS,
    PRIMARY_ROTVEC_COLUMNS,
    export_layer3_window,
    export_window_for_jvcpca,
)
from pre_jvcpca_review.feature_scope import load_feature_scope
from pre_jvcpca_review.joint_overlap import (
    canonical_names_to_link_tuples,
    classify_links,
    emit_joint_comparability_warnings,
    non_comparable_required_features,
    overlap_dataframe,
    write_joint_overlap_table,
)
from pre_jvcpca_review.load_layer2 import load_link_manifest
from pre_jvcpca_review.notebook_ui import (
    JointSelector,
    display_summary_card,
    display_table,
)
from pre_jvcpca_review.canonical_manifest import (
    DEFAULT_PILOT_MANIFEST,
    ManifestError,
    load_pilot_manifest,
    pilot_feature_order,
    pilot_link_order,
    resolve_session_links_from_manifest,
)
from pre_jvcpca_review.build import build_full_review, build_mapping_only
from pre_jvcpca_review.pairing import run_pairing_gate
from pre_jvcpca_review.review_display import display_review_tables
from pre_jvcpca_review.review_output import (
    participant_out_dir,
    require_review_context,
    resolve_review_out_dir,
)
from pre_jvcpca_review.session_index import (
    DEFAULT_LAYER1_ROOT,
    DEFAULT_LAYER2_ROOT,
    build_session_index,
    participants,
    session_row,
    sessions_for,
    write_session_index,
)
from pre_jvcpca_review.warnings import WarningCollector

WIDE = widgets.Layout(width="98%")
BTN = widgets.Layout(width="220px", margin="4px")

# --- Default input ROOTS (editable, but these are the defaults) ---
LAYER1_ROOT = widgets.Text(value=str(DEFAULT_LAYER1_ROOT), description="L1 root:", layout=WIDE)
LAYER2_ROOT = widgets.Text(value=str(DEFAULT_LAYER2_ROOT), description="L2 root:", layout=WIDE)

# Resolved per-session dirs (auto-filled after participant + session selection).
LAYER1_DIR = widgets.Text(value="", description="L1 session:", layout=WIDE, disabled=True)
LAYER2_DIR = widgets.Text(value="", description="L2 session:", layout=WIDE, disabled=True)

# --- Default feature scope (config-controlled: core, fingers excluded) ---
SCOPE = load_feature_scope()
PILOT_MANIFEST = load_pilot_manifest(DEFAULT_PILOT_MANIFEST)
CANONICAL_FEATURE_ORDER = pilot_feature_order(PILOT_MANIFEST)
REQUIRED_LINKS = pilot_link_order(PILOT_MANIFEST)

# Discovery / selection state.
SESSION_INDEX: pd.DataFrame | None = None
CURRENT_OVERLAP: pd.DataFrame | None = None
CURRENT_ROW = None
EXPORT_BLOCKED = True  # default-safe: export disabled until checks pass

DATADESCRIPTIONS_PATH = widgets.Text(
    value=str(PROJECT_ROOT / "reevluate_project" / "671_T1_P1_R1_Take 2026-01-06 03.57.12 PM_001_DataDescriptions.csv"),
    description="DataDesc:",
    layout=WIDE,
)
OUTPUT_DIR = widgets.Text(
    value=str(PROJECT_ROOT / "outputs" / "pre_jvcpca_review"),
    description="Review root:",
    layout=WIDE,
)
FRAME_START = widgets.IntText(value=16000, description="Start frame:", layout=widgets.Layout(width="200px"))
FRAME_END = widgets.IntText(value=17000, description="End frame:", layout=widgets.Layout(width="200px"))
WINDOW_LABEL = widgets.Text(value="window_01", description="Window label:", layout=widgets.Layout(width="320px"))

# Participant / session selectors (populated by discovery).
PARTICIPANT_DD = widgets.Dropdown(options=[], description="Participant:", layout=widgets.Layout(width="320px"))
SESSION_DD = widgets.Dropdown(options=[], description="Session:", layout=widgets.Layout(width="420px"))

QC_GAP_0P5 = widgets.Checkbox(value=True, description="gap ≥ 0.5 s")
QC_GAP_0P2 = widgets.Checkbox(value=True, description="gap ≥ 0.2 s")
QC_ARTIFACT = widgets.Checkbox(value=True, description="artifact_sigma")
QC_SWAP = widgets.Checkbox(value=True, description="segment_swap")
ALLOW_NAN_MATRIX = widgets.Checkbox(
    value=False,
    description="Allow NaN in JcvPCA matrix (do not impute)",
    tooltip=(
        "Unchecked (default): export fails if any filtered-analysis rotvec is NaN. "
        "Checked: export proceeds and keeps NaNs in the matrix; counts are recorded in the manifest. "
        "Does not fill or replace missing values."
    ),
)

status_html = widgets.HTML(value="<i>Ready — click <b>Discover participants/sessions</b> to scan the roots.</i>")
warning_html = widgets.HTML(value="")
joint_selector: JointSelector | None = None
out_discover = widgets.Output()
out_pairing = widgets.Output()
out_overlap = widgets.Output()
out_warnings = widgets.Output()
out_mapping = widgets.Output()
out_review = widgets.Output()
out_export = widgets.Output()
out_tables = widgets.Output()


def _participant_out_dir(participant_id: str) -> Path:
    return participant_out_dir(Path(OUTPUT_DIR.value), participant_id)


def _review_out_dir() -> Path:
    if CURRENT_ROW is None or not SESSION_DD.value:
        return Path(OUTPUT_DIR.value) / "_unset"
    return resolve_review_out_dir(
        Path(OUTPUT_DIR.value),
        str(CURRENT_ROW["participant_id"]),
        str(SESSION_DD.value),
        str(WINDOW_LABEL.value),
    )


def _dd_path() -> Path | None:
    path = Path(DATADESCRIPTIONS_PATH.value)
    return path if path.is_file() else None


def _require_review_context() -> str | None:
    return require_review_context(
        current_row=CURRENT_ROW,
        layer1_dir=LAYER1_DIR.value,
        layer2_dir=LAYER2_DIR.value,
    )


def _severity_color(sev: str) -> str:
    return {
        "blocking": "#b00020",
        "strong_warning": "#d35400",
        "warning": "#b8860b",
        "info": "#2e7d32",
    }.get(sev, "#444")


def _render_warning_panel(collector: WarningCollector) -> None:
    """Render compact summary + detailed table, and set EXPORT_BLOCKED."""
    global EXPORT_BLOCKED
    summary = collector.summary()
    EXPORT_BLOCKED = bool(summary["has_blocking"])
    btn_export.disabled = EXPORT_BLOCKED
    if EXPORT_BLOCKED:
        verdict = "<span style='color:#b00020;font-weight:bold'>EXPORT BLOCKED</span>"
    elif summary["requires_user_approval"]:
        verdict = "<span style='color:#d35400;font-weight:bold'>EXPORT ALLOWED — strong warnings need review</span>"
    else:
        verdict = "<span style='color:#2e7d32;font-weight:bold'>EXPORT ALLOWED</span>"
    warning_html.value = (
        f"{verdict} &nbsp; | &nbsp; blocking: <b>{summary['n_blocking']}</b>, "
        f"strong: <b>{summary['n_strong_warning']}</b>, "
        f"warning: <b>{summary['n_warning']}</b>, info: <b>{summary['n_info']}</b>"
    )
    out_warnings.clear_output(wait=True)
    with out_warnings:
        df = collector.to_dataframe()
        if df.empty:
            display(HTML("<p style='color:#2e7d32'>No warnings.</p>"))
        else:
            show = df[["severity", "category", "warning_id", "message", "recommended_action"]]
            display_table(show, "window_warnings (preview)", max_rows=50)


def _selected_qc_types() -> list[str]:
    types = []
    if QC_GAP_0P5.value:
        types.append("gap_0p5")
    if QC_GAP_0P2.value:
        types.append("gap_0p2")
    if QC_ARTIFACT.value:
        types.append("artifact_sigma")
    if QC_SWAP.value:
        types.append("segment_swap")
    return types


def on_discover(_btn=None) -> None:
    global SESSION_INDEX
    out_discover.clear_output(wait=True)
    with out_discover:
        try:
            SESSION_INDEX = build_session_index(Path(LAYER1_ROOT.value), Path(LAYER2_ROOT.value))
        except Exception as exc:
            display(HTML(f"<p style='color:red'>Discovery failed: {exc}</p>"))
            return
        if SESSION_INDEX.empty:
            display(HTML("<p style='color:red'>No sessions found under the given roots.</p>"))
            return
        base = Path(OUTPUT_DIR.value).parent
        base.mkdir(parents=True, exist_ok=True)
        write_session_index(SESSION_INDEX, base / "session_index.csv")
        pids = participants(SESSION_INDEX)
        PARTICIPANT_DD.options = pids
        n_matched = int(SESSION_INDEX["is_matched"].sum())
        display(HTML(
            f"<p><b>{len(SESSION_INDEX)}</b> sessions discovered for participants "
            f"{pids} — <b>{n_matched}</b> matched. Saved <code>session_index.csv</code>.</p>"
        ))
        cols = ["session_id", "is_matched", "n_frames_layer1", "n_frames_layer2", "match_warning"]
        display_table(SESSION_INDEX[cols], "session_index.csv", max_rows=50)
        if pids:
            PARTICIPANT_DD.value = pids[0]
            on_participant_change()


def on_participant_change(_change=None) -> None:
    global CURRENT_OVERLAP
    if SESSION_INDEX is None or not PARTICIPANT_DD.value:
        return
    pid = str(PARTICIPANT_DD.value)
    sess_df = sessions_for(SESSION_INDEX, pid)
    out_pairing.clear_output(wait=True)
    with out_pairing:
        cols = ["session_id", "timepoint", "part_id", "repetition_id", "is_matched", "match_warning"]
        display_table(sess_df[cols], f"Layer1/Layer2 pairing — participant {pid}", max_rows=50)

    # Cross-session joint overlap / comparability (matched sessions only).
    matched = sess_df[sess_df["is_matched"]]
    sess_links = {}
    for _, r in matched.iterrows():
        if r["layer2_run_dir"]:
            try:
                lp = resolve_layer2(r["layer2_run_dir"])
                sess_links[r["session_id"]] = load_link_manifest(lp.link_manifest)
            except Exception:
                pass
    out_overlap.clear_output(wait=True)
    with out_overlap:
        if len(sess_links) >= 1:
            rows = classify_links(sess_links, candidate_links=REQUIRED_LINKS)
            CURRENT_OVERLAP = overlap_dataframe(rows, pid, list(sess_links))
            out_dir = _participant_out_dir(pid)
            write_joint_overlap_table(CURRENT_OVERLAP, out_dir / "joint_overlap_table.csv")
            display(HTML(
                f"<p>Joint overlap across <b>{len(sess_links)}</b> session(s); saved "
                f"<code>{out_dir / 'joint_overlap_table.csv'}</code></p>"
            ))
            show = CURRENT_OVERLAP[[
                "canonical_link_name", "classification", "present_in_sessions",
                "missing_in_sessions", "recommended_action",
            ]]
            display_table(show, "joint_overlap_table.csv (required canonical links)", max_rows=50)
            bad = non_comparable_required_features(CURRENT_OVERLAP, REQUIRED_LINKS)
            if bad:
                display(HTML(
                    "<p style='color:#b00020'><b>Not directly comparable across sessions:</b> "
                    f"{', '.join(bad)} — cross-session Layer 3 export would be blocked.</p>"
                ))
        else:
            CURRENT_OVERLAP = None
            display(HTML("<p style='color:#b8860b'>No matched Layer 2 sessions to compare.</p>"))

    SESSION_DD.options = sess_df["session_id"].tolist()
    if SESSION_DD.options:
        SESSION_DD.value = SESSION_DD.options[0]
        on_session_change()


def _warning_identity_kw() -> dict[str, str]:
    if CURRENT_ROW is None:
        return {}
    return {
        "participant_id": str(CURRENT_ROW["participant_id"]),
        "session_id": str(CURRENT_ROW["session_id"]),
    }


def _selected_scope_links() -> list[tuple[str, str]]:
    if joint_selector is None:
        return []
    return canonical_names_to_link_tuples(joint_selector.selected_canonical_names())


def _export_comparability_scope() -> list[tuple[str, str]] | None:
    """Scope for comparability checks. Empty checkbox selection → pilot required links."""
    selected = _selected_scope_links()
    if selected:
        return selected
    return list(REQUIRED_LINKS)


def _collect_session_warnings() -> WarningCollector:
    """Pairing + cross-session comparability warnings scoped to export comparability."""
    collector = WarningCollector()
    if CURRENT_ROW is not None:
        run_pairing_gate(CURRENT_ROW, collector)
    if CURRENT_OVERLAP is not None:
        emit_joint_comparability_warnings(
            collector,
            CURRENT_OVERLAP,
            _export_comparability_scope() or [],
            **_warning_identity_kw(),
        )
    return collector


def _refresh_warnings(_=None) -> None:
    if CURRENT_ROW is None:
        return
    _render_warning_panel(_collect_session_warnings())


def on_session_change(_change=None) -> None:
    global CURRENT_ROW
    if SESSION_INDEX is None or not SESSION_DD.value:
        return
    row = session_row(SESSION_INDEX, str(SESSION_DD.value))
    CURRENT_ROW = row
    LAYER1_DIR.value = str(row["layer1_run_dir"] or "")
    LAYER2_DIR.value = str(row["layer2_run_dir"] or "")
    reload_joint_list()
    _refresh_warnings()


def _overlap_classifications() -> dict[str, str]:
    if CURRENT_OVERLAP is None:
        return {}
    return {
        str(row["canonical_link_name"]): str(row["classification"])
        for _, row in CURRENT_OVERLAP.iterrows()
    }


def _default_joint_link_ids(links) -> set[str]:
    try:
        pilot_ids, _ = resolve_session_links_from_manifest(PILOT_MANIFEST, links)
        return set(pilot_ids[:3])
    except ManifestError:
        core = [link.link_id for link in links if link.feature_scope == "core_candidate"]
        return set(core[:3])


def reload_joint_list(_=None) -> None:
    global joint_selector
    try:
        links = load_link_manifest(resolve_layer2(Path(LAYER2_DIR.value)).link_manifest)
    except Exception as exc:
        status_html.value = f"<span style='color:red'>Could not load joints: {exc}</span>"
        return
    joint_selector = JointSelector.from_links(
        links,
        default_link_ids=_default_joint_link_ids(links),
        canonical_order=REQUIRED_LINKS,
        overlap_by_canonical=_overlap_classifications(),
    )
    joint_selector.on_selection_change(_refresh_warnings)
    n_core = sum(1 for link in links if link.feature_scope == "core_candidate")
    status_html.value = (
        f"<b>{len(links)}</b> joints loaded from Layer 2 folder "
        f"(<b>{n_core}</b> core_candidate). Labels use <code>canonical_link_name</code> "
        f"(same as <code>joint_overlap_table.csv</code>)."
    )
    joint_panel.children = (
        widgets.HTML(
            "<b>Select joints for review</b> "
            "(<code>canonical_link_name</code>; matches joint_overlap_table.csv)"
        ),
        joint_selector.filter_core_only,
        widgets.HBox([btn_select_core, btn_clear_joints]),
        joint_selector.container,
    )


def on_select_core(_btn) -> None:
    if joint_selector:
        joint_selector.select_core_candidates()
        _refresh_warnings()


def on_clear_joints(_btn) -> None:
    if joint_selector:
        joint_selector.clear_selection()
        _refresh_warnings()


def on_run_mapping(_btn) -> None:
    out_mapping.clear_output(wait=True)
    ctx_err = _require_review_context()
    if ctx_err:
        with out_mapping:
            display(HTML(f"<p style='color:red'>{ctx_err}</p>"))
        return
    if not joint_selector:
        with out_mapping:
            display(HTML("<p style='color:red'>Load joints first (Reload joint list).</p>"))
        return
    selected_ids = joint_selector.selected_link_ids()
    selected_names = joint_selector.selected_canonical_names()
    if not selected_ids:
        with out_mapping:
            display(HTML("<p style='color:red'>Select at least one joint checkbox first.</p>"))
        return
    out = _review_out_dir()
    out.mkdir(parents=True, exist_ok=True)
    dd = _dd_path()
    with out_mapping:
        try:
            mapping_path = build_mapping_only(
                Path(LAYER1_DIR.value),
                Path(LAYER2_DIR.value),
                out,
                dd,
                selected_link_ids=selected_ids,
            )
            df = pd.read_csv(mapping_path)
            dd_note = (
                f"DataDescriptions: <code>{dd}</code>"
                if dd
                else "<span style='color:#b8860b'>DataDescriptions not found — heuristic mapping only</span>"
            )
            display(HTML(
                f"<p><b>{len(df)} markers/regions</b> mapped to candidate links for "
                f"selected joints: {', '.join(selected_names)}</p>"
                f"<p>{dd_note}</p>"
                f"<p>Output: <code>{out}</code></p>"
            ))
            display_table(df, "mapping_logic_table.csv (joint-scoped)")
            warn_path = out / "window_warnings.csv"
            if warn_path.is_file():
                display_table(pd.read_csv(warn_path), "window_warnings.csv", max_rows=20)
        except Exception as exc:
            display(HTML(f"<p style='color:red'>{exc}</p>"))


def on_run_review(_btn) -> None:
    out_review.clear_output(wait=True)
    ctx_err = _require_review_context()
    if ctx_err:
        with out_review:
            display(HTML(f"<p style='color:red'>{ctx_err}</p>"))
        return
    if not joint_selector:
        with out_review:
            display(HTML("<p style='color:red'>Load joints first (Reload joint list).</p>"))
        return
    selected_ids = joint_selector.selected_link_ids()
    selected_names = joint_selector.selected_canonical_names()
    if not selected_ids:
        with out_review:
            display(HTML("<p style='color:red'>Select at least one joint checkbox.</p>"))
        return
    qc = _selected_qc_types()
    if not qc:
        with out_review:
            display(HTML("<p style='color:red'>Select at least one QC evidence type.</p>"))
        return
    out = _review_out_dir()
    out.mkdir(parents=True, exist_ok=True)
    dd = _dd_path()
    with out_review:
        try:
            paths = build_full_review(
                layer1_dir=Path(LAYER1_DIR.value),
                layer2_dir=Path(LAYER2_DIR.value),
                out_dir=out,
                frame_start=int(FRAME_START.value),
                frame_end=int(FRAME_END.value),
                selected_link_ids=selected_ids,
                qc_evidence=qc,
                datadescriptions=dd,
            )
            summary = pd.read_csv(paths["window_decision_summary.csv"]).iloc[0]
            dd_status = (
                f"found/used (<code>{summary.get('mapping_mode', '')}</code>)"
                if summary.get("datadescriptions_used")
                else "not found — heuristic mapping"
            )
            display(HTML(
                f"<p><b>Review complete.</b> {len(selected_ids)} joints, frames "
                f"{FRAME_START.value}–{FRAME_END.value}.</p>"
                f"<p>Selected: {', '.join(selected_names)}</p>"
                f"<p>DataDescriptions: {dd_status}</p>"
                f"<p>Output: <code>{out}</code></p>"
                f"<p>Wrote: {', '.join(sorted(paths.keys()))}</p>"
            ))
        except Exception as exc:
            display(HTML(f"<p style='color:red'>{exc}</p>"))


def on_export_window(_btn) -> None:
    """Layer 3-safe canonical export only. Disabled while blocking warnings exist."""
    out_export.clear_output(wait=True)
    if CURRENT_ROW is None:
        with out_export:
            display(HTML("<p style='color:red'>Select a participant and session first.</p>"))
        return
    out = _review_out_dir()
    scope = _selected_scope_links()
    with out_export:
        try:
            result = export_layer3_window(
                Path(LAYER1_DIR.value),
                Path(LAYER2_DIR.value),
                out,
                int(FRAME_START.value),
                int(FRAME_END.value),
                session_row=CURRENT_ROW,
                window_label=str(WINDOW_LABEL.value),
                allow_nan_matrix=ALLOW_NAN_MATRIX.value,
                overlap_df=CURRENT_OVERLAP,
                # Empty checkbox selection → use pilot required links (same as preview gate).
                scope_required_links=scope if scope else None,
            )
        except Exception as exc:
            display(HTML(f"<p style='color:red'>Export error: {exc}</p>"))
            return

        if result["status"] == "blocked":
            display(HTML(
                "<p style='color:#b00020'><b>Export blocked by blocking warnings.</b> "
                f"See <code>{result['warnings_csv']}</code>. No matrix written.</p>"
            ))
            display_table(pd.read_csv(result["warnings_csv"]), "window_warnings.csv", max_rows=50)
            return

        import json

        paths = result["paths"]
        manifest = json.loads(Path(paths["manifest"]).read_text(encoding="utf-8"))
        matrix_df = pd.read_parquet(paths["jvcpca_matrix"])
        feature_cols = [c for c in matrix_df.columns if c not in MATRIX_IDENTITY_COLUMNS]
        display(HTML(
            f"<p><b>Layer 3-safe export complete.</b> "
            f"layer3_safe=<b>{manifest['layer3_safe']}</b>, "
            f"policy=<code>{manifest['feature_naming_policy']}</code></p>"
            f"<p>Output dir: <code>{out}</code></p>"
            f"<p>Matrix shape: <b>{manifest['jvcpca_matrix_row_count']} × "
            f"{manifest['jvcpca_matrix_column_count']}</b> · "
            f"features: <b>{manifest['n_features']}</b> · "
            f"NaN count: <b>{manifest['nan_count_matrix']}</b></p>"
            f"<p><b>Canonical feature order (first 6):</b> "
            f"{', '.join(feature_cols[:6])} …</p>"
        ))
        preview_cols = MATRIX_IDENTITY_COLUMNS + feature_cols[: min(6, len(feature_cols))]
        display_table(matrix_df[preview_cols], "window_jvcpca_matrix.parquet (preview)", max_rows=8)
        if manifest.get("requires_user_approval"):
            display(HTML(
                "<p style='color:#d35400'><b>Strong warnings present</b> — review "
                "<code>window_warnings.csv</code> before using this export for Layer 3.</p>"
            ))


def on_show_tables(_btn) -> None:
    out_tables.clear_output(wait=True)
    ctx_err = _require_review_context()
    with out_tables:
        if ctx_err:
            display(HTML(f"<p style='color:red'>{ctx_err}</p>"))
            return
        display_review_tables(_review_out_dir())


btn_reload = widgets.Button(description="Reload joint list", button_style="info", layout=BTN)
btn_select_core = widgets.Button(description="Select core joints", layout=BTN)
btn_clear_joints = widgets.Button(description="Clear joint selection", layout=BTN)
btn_mapping = widgets.Button(description="Run mapping table", button_style="primary", layout=BTN)
btn_review = widgets.Button(description="Run full review", button_style="success", layout=BTN)
btn_export = widgets.Button(description="Export Layer 3 window", button_style="info", layout=BTN, disabled=True)
btn_tables = widgets.Button(description="Show review tables", button_style="warning", layout=BTN)

btn_discover = widgets.Button(
    description="Discover participants/sessions", button_style="primary",
    layout=widgets.Layout(width="260px", margin="4px"),
)
btn_preview = widgets.Button(
    description="Preview warnings", button_style="warning",
    layout=widgets.Layout(width="200px", margin="4px"),
)


def on_preview_warnings(_btn=None) -> None:
    if CURRENT_ROW is None:
        warning_html.value = "<span style='color:red'>Select a participant + session first.</span>"
        return
    _render_warning_panel(_collect_session_warnings())


btn_discover.on_click(on_discover)
btn_preview.on_click(on_preview_warnings)
PARTICIPANT_DD.observe(on_participant_change, names="value")
SESSION_DD.observe(on_session_change, names="value")
btn_reload.on_click(reload_joint_list)
btn_select_core.on_click(on_select_core)
btn_clear_joints.on_click(on_clear_joints)
btn_mapping.on_click(on_run_mapping)
btn_review.on_click(on_run_review)
btn_export.on_click(on_export_window)
btn_tables.on_click(on_show_tables)

joint_panel = widgets.VBox([])

SCOPE_HTML = widgets.HTML(
    "<div style='background:#f5f7fa;border:1px solid #dde;padding:6px 10px;'>"
    f"<b>Default feature scope</b> (from <code>{SCOPE.source_path.name}</code>): "
    f"scope=<code>{SCOPE.default_body_scope}</code>, "
    f"link set=<code>{SCOPE.core_link_set_name}</code>, "
    f"exclude_fingers=<b>{SCOPE.exclude_fingers}</b>, exclude_toes=<b>{SCOPE.exclude_toes}</b>, "
    f"naming=<code>{SCOPE.feature_naming_policy}</code><br>"
    f"<b>{len(REQUIRED_LINKS)}</b> canonical links / <b>{len(CANONICAL_FEATURE_ORDER)}</b> features. "
    "Change scope in the config file, not in notebook cells."
    "</div>"
)

display(
    widgets.VBox([
        widgets.HTML("<h2>1. Input roots</h2>"),
        LAYER1_ROOT,
        LAYER2_ROOT,
        OUTPUT_DIR,
        btn_discover,
        out_discover,
        widgets.HTML("<h2>2. Participant &amp; session</h2>"),
        PARTICIPANT_DD,
        widgets.HTML("<b>Layer 1 / Layer 2 pairing</b>"),
        out_pairing,
        widgets.HTML("<b>Joint / link overlap &amp; comparability</b>"),
        out_overlap,
        SESSION_DD,
        LAYER1_DIR,
        LAYER2_DIR,
        widgets.HTML("<h2>3. Window &amp; feature scope</h2>"),
        widgets.HBox([FRAME_START, FRAME_END]),
        WINDOW_LABEL,
        SCOPE_HTML,
        widgets.HTML("<b>QC evidence types</b>"),
        widgets.HBox([QC_GAP_0P5, QC_GAP_0P2, QC_ARTIFACT, QC_SWAP]),
        ALLOW_NAN_MATRIX,
        status_html,
        joint_panel,
        widgets.HTML("<h2>4. Warnings (shown before export)</h2>"),
        btn_preview,
        warning_html,
        out_warnings,
        widgets.HTML("<h2>5. Export (canonical Layer 3-safe only)</h2>"),
        widgets.HBox([btn_export]),
        out_export,
        widgets.HTML("<hr><h3>Diagnostics (legacy review path)</h3>"),
        widgets.HBox([btn_reload, btn_mapping, btn_review, btn_tables]),
        widgets.HTML("<h4>Mapping output</h4>"),
        out_mapping,
        widgets.HTML("<h4>Review status</h4>"),
        out_review,
        widgets.HTML("<h4>Review tables</h4>"),
        out_tables,
    ])
)